In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("HF_TOKEN")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"HF_TOKEN loaded ({len(token)} characters): {masked}")
else:
    print("HF_TOKEN not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'HF_TOKEN', and load_dotenv() ran without error.")

HF_TOKEN loaded (37 characters): hf_k...ivRP


In [2]:
"""
Few-shot relevance classification using Llama 3-8B-Instruct.

Same crash-safe / resumable design as classify_with_open_llm.py.

NOTE: This is a GATED model on Hugging Face. Using Llama 3 (not 3.1) here
specifically because access to the Llama 3.1 gating group is pending at time
of writing; Llama 3 access was already approved. Once your 3.1 request
clears, you can switch MODEL_NAME below to
"meta-llama/Llama-3.1-8B-Instruct" if you'd rather compare the newer
generation -- both are real, appropriately-sized options for a single 16GB
GPU. Either way, you must run `huggingface-cli login` (or set the HF_TOKEN
environment variable) with an approved account before this will download.

Meta has since announced newer Llama generations (Llama 4, Llama 5), but
their flagship variants are much larger than fits on a single 16GB GPU.
"""

import json
import os
import re
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Pass the HF token explicitly rather than relying on huggingface-cli login
# having taken effect in this exact process -- browser-granted access and a
# CLI login in one terminal do NOT automatically authenticate a separate
# Jupyter kernel or script process. Set this as an environment variable
# (export HF_TOKEN=...) before running, or paste your token directly here
# (only if this file stays private and is never committed/shared).
from huggingface_hub import HfFolder

# Pass the HF token explicitly rather than relying on huggingface-cli login
# having taken effect in this exact process -- browser-granted access and a
# CLI login in one terminal do NOT automatically authenticate a separate
# Jupyter kernel or script process. Tries the HF_TOKEN environment variable
# first, then falls back to whatever `hf auth login` already saved locally.
HF_TOKEN = os.environ.get("HF_TOKEN") or HfFolder.get_token()
if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found (checked HF_TOKEN env var and the "
        "local CLI login). Run `hf auth login` in this exact terminal, or "
        "`export HF_TOKEN=your_token` before running the script."
    )

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  # using this since your
    # Llama 3.1 access request is still pending; once approved, you can
    # switch this to "meta-llama/Llama-3.1-8B-Instruct" if you want the
    # newer generation for the comparison instead
PROMPT_FILE = "gpt_classification_prompt.txt"
INPUT_FILE = "validation_results.xlsx"
OUTPUT_JSONL = "classification_results_llama.jsonl"
MAX_NEW_TOKENS = 200


def load_system_prompt(path: str) -> str:
    with open(path) as f:
        return f.read()


def load_input_papers(path: str) -> pd.DataFrame:
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def already_processed_ids(output_path: str) -> set:
    if not os.path.exists(output_path):
        return set()
    ids = set()
    with open(output_path) as f:
        for line in f:
            try:
                ids.add(json.loads(line)["wos_id"])
            except (json.JSONDecodeError, KeyError):
                continue
    return ids


def extract_json(text: str) -> dict | None:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def build_user_message(title: str, abstract: str) -> str:
    return f"Title: {title}\nAbstract: {abstract}"


def main():
    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
        token=HF_TOKEN,
    )
    model.eval()

    system_prompt = load_system_prompt(PROMPT_FILE)
    papers = load_input_papers(INPUT_FILE)
    done_ids = already_processed_ids(OUTPUT_JSONL)
    print(f"Total papers: {len(papers)} | Already processed: {len(done_ids)}")

    n_ok, n_parse_failed = 0, 0
    flagged = []

    with open(OUTPUT_JSONL, "a") as out_f:
        for row in papers.itertuples():
            wos_id = row.wos_id
            if wos_id in done_ids:
                continue

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": build_user_message(row.title, row.abstract)},
            ]
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(model.device)

            with torch.no_grad():
                output = model.generate(
                    inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    repetition_penalty=1.3,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                )
            generated = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

            parsed = extract_json(generated)
            result = {"wos_id": wos_id, "title": row.title, "raw_model_output": generated}
            if parsed is not None and "label" in parsed:
                result.update(parsed)
                n_ok += 1
            else:
                result.update({"label": None, "trap_reason": None, "reason": None, "parse_failed": True})
                n_parse_failed += 1
                flagged.append(wos_id)

            out_f.write(json.dumps(result) + "\n")
            out_f.flush()

    print(f"\nDone. OK: {n_ok} | Parse failed: {n_parse_failed}")
    if flagged:
        print(f"Flagged wos_ids: {flagged[:20]}{' ...' if len(flagged) > 20 else ''}")
    print(f"Results saved to {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (c:\Users\olagunju\AppData\Local\anaconda3\envs\Shola-environment\Lib\site-packages\huggingface_hub\__init__.py)